In [1]:
from datasets import load_dataset

ds = load_dataset("TeichAI/claude-haiku-4.5-high-reasoning-1700x")

/home/hardik-mutha/Desktop/Code/5Guys1Code_Techfiesta/server/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


In [30]:
print(ds.keys())        # e.g. train, test
print(ds["train"])      # view train split info

dict_keys(['train'])
Dataset({
    features: ['messages'],
    num_rows: 1688
})


In [3]:
import re

def is_garbage(text):
    # Rule 1: If it has a code block (```), it's a coding task. Dump it.
    if "```" in text:
        return True
        
    # Rule 2: If it has >3 curly braces, it's C/Java/JSON. Dump it.
    if text.count("{") + text.count("}") > 3:
        return True
        
    # Rule 3: If it's too short (< 50 chars), it's not a useful essay.
    if len(text) < 50:
        return True
        
    return False

def quick_clean(text):
    # Remove <think> tags
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    
    # Remove basic markdown (**bold**, ## header, [links])
    text = text.replace("**", "").replace("##", "").replace("`", "")
    
    return " ".join(text.split())

# --- RUN IT ---
df = ds['train'].to_pandas()
final_data = []

for row in df["messages"]:
    raw_text = row[1]['content']
    
    # 1. Check if it's trash
    if is_garbage(raw_text):
        continue
        
    # 2. Clean it
    clean_text = quick_clean(raw_text)
    
    # 3. Save it
    final_data.append({
        "text": clean_text,
        "label": 1
    })

print(f"Kept {len(final_data)} clean text rows.")

Kept 764 clean text rows.


In [4]:
import pandas as pd

final_df = pd.DataFrame(final_data)
final_df.to_csv("claude-haiku-4.5-high-reasoning.csv", index=False)

In [7]:
# read a csv and swap its columns
import pandas as pd
data = pd.read_csv("../../dataset/llama_falcon_combined_formatted.csv")
data

,text,model,label
0,"Dear Principal,\n\nI am writing to express my ...",falcon_180b_v1,1
1,When people are faced with a difficult decisio...,falcon_180b_v1,1
2,"As a grade 12 student, I believe that summer p...",falcon_180b_v1,1
3,"Dear Principal,\n\nI am writing to share my th...",falcon_180b_v1,1
4,"""Making Mona Lisa Smile"" is an interesting art...",falcon_180b_v1,1
...,...,...,...
10394,"Driverless cars, also known as autonomous cars...",falcon_180b,1
10395,Driverless Cars: A Necessity for Our Future\n\...,falcon_180b,1
10396,The Pros and Cons of Driverless Cars\n\nThe wo...,falcon_180b,1
10397,The development of driverless cars has been a ...,falcon_180b,1


In [ ]:
temp = data['model'].copy()
data['model'] = data['label']
data['label'] = temp
data = data.rename(columns={
    "model": "label",
    "label": "model"
})


In [10]:
data.to_csv("../../dataset/llama_falcon_combined_formatted.csv", index=False)

In [2]:
import pandas as pd

df = pd.read_csv("../../dataset/persuade_corpus_1.0.csv")
sampled = df.sample(n=12000, random_state=42)
print(sampled.columns)
sampled.drop(columns=["essay_id_comp","competition_set","discourse_id","discourse_type","discourse_text","discourse_start","discourse_end","discourse_type_num"],inplace=True)
sampled["model"] = "human"
sampled["label"] = 1
sampled.rename(columns={"full_text": "text"}, inplace=True)

Index(['essay_id_comp', 'competition_set', 'full_text', 'discourse_id',
       'discourse_start', 'discourse_end', 'discourse_text', 'discourse_type',
       'discourse_type_num'],
      dtype='str')


In [3]:
sampled.rename(columns={"full_text": "text"}, inplace=True)
sampled.to_csv("../../dataset/persuade_corpus_1.0_sampled.csv", index=False)

In [9]:
import pandas as pd
data1 = pd.read_csv("../../dataset/llama_falcon_combined_formatted.csv")
data2 = pd.read_csv("../../dataset/claude-haiku-4.5-high-reasoning.csv")
data3 = pd.read_csv("../../dataset/gemini_3_pro.csv")
data4 = pd.read_csv("../../dataset/claude_haiku_4.5_opus.csv")
data5 = pd.read_csv("../../dataset/claude_4.5_opus.csv")
data6 = pd.read_csv("../../dataset/daigt_formatted.csv")
data7 = pd.read_csv("../../dataset/persuade_corpus_1.0_sampled.csv")
data8 = pd.read_csv("../../dataset/raid_attacks_200mb.csv")

data8.drop(columns=["attack"], inplace=True)


# combine all dataframes
data = pd.concat([data1, data2, data3, data4, data5, data6, data7, data8], ignore_index=True)
data.to_csv("../../dataset/combined_dataset.csv", index=False)

In [7]:
data7 = pd.read_csv("../../dataset/persuade_corpus_1.0_sampled.csv")
data7.head()

,text,model,label
0,There have been a few close calls during our p...,human,1
1,I think that they should get rid of the electo...,human,1
2,Ever been rushed around? Have had so much to d...,human,1
3,Hello! Today you problem feel super board with...,human,1
4,No texting allowed while driving.\n\nEverywher...,human,1


In [4]:
import pandas as pd
data8 = pd.read_csv("../../dataset/raid_attacks_200mb.csv")

data8.drop(columns=["attack"], inplace=True)

In [8]:
data8.head()

,text,label,model
0,A class of b-theoric questions known as Inequa...,1,gpt2
1,\n\nThe Lost World: Jurassic Park is a 1997 ...,1,gpt3
2,Ingredients:\n\n1/2 cup (60 ml) milk\n\n3 tabl...,1,gpt2
3,"In the novel ""Shattered Peace,"" the story is c...",1,chatgpt
4,F​o​r​m​e​r​ ​E​n​g​l​a​n​d​ ​U​n​d​e​r​-​2​1​...,1,mpt


In [17]:
final_data = pd.read_csv("../../dataset/combined_dataset.csv")
final_data.head()

,text,label,model
0,I used to be friends with this man who was a m...,1,gpt3
1,Have you ever been extremely tired or bored? W...,0,human
2,"In the small town of Millfield, nestled In The...",1,mistral-chat
3,"Іt wаs rеlеаsеd іn 1999, аnd іt quісklу bесаmе...",1,gpt2
4,\n\nImage registration is the process of align...,1,gpt3


In [15]:
import pandas as pd

final_data= final_data.sample(frac=1, random_state=42).reset_index(drop=True)

final_data.to_csv("../../dataset/combined_dataset.csv", index=False)

In [ ]:
import pandas as pd
import re

df = pd.read_csv("./combined_dataset.csv")

def clean_text(text):
    if isinstance(text, str):
        text = re.sub(r'[\u200B-\u200D\uFEFF]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
    return text

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(clean_text)

df.to_csv("combined_dataset_1.csv", index=False)

In [13]:
data = pd.read_csv("../../dataset/output.csv")
len(data)

183299